# Wardrobe Combinations — Image Generator

Generates photorealistic menswear images for all 154 wardrobe combinations using **PixArt-α**.

**Setup:** Runtime > Change runtime type > **T4 GPU**

In [ ]:
# Cell 1: Install dependencies
!pip install -q diffusers transformers accelerate safetensors torch sentencepiece

In [ ]:
# Cell 2: Upload wardrobe_mongo.json
import json
from google.colab import files

uploaded = files.upload()  # Upload wardrobe_mongo.json when prompted

with open('wardrobe_mongo.json', 'r') as f:
    combinations = json.load(f)

print(f'Loaded {len(combinations)} combinations')
print(f'Categories: {set(c["input_json"]["category"] for c in combinations)}')
print(f'Sample: {combinations[0]["combo_id"]}')

In [ ]:
# Cell 3: Prompt template & assembly function
# PixArt-α uses T5 text encoder — supports up to 512 tokens (our prompt is ~440).

PROMPT_TEMPLATE = """Photorealistic premium commercial menswear photograph of an adult male model wearing a sophisticated business formal outfit.

MODEL & FRAMING
Adult male model with a natural, realistic physique. Face intentionally cropped out; do not show the full face. Frame the model from approximately the lower neck and chin area down to the shoes. Full outfit must remain visible. Upright, relaxed posture. Both hands placed naturally inside the trouser pockets. Legs positioned in a relaxed fashion-model stance: one leg crossed naturally in front of the other at the ankles. Natural human proportions and anatomy.

SHIRT
{shirt_color} ({shirt_hex}) long-sleeve dress shirt in fine cotton with subtle woven texture. Modern tailored fit. Semi-spread collar. Sleeves rolled neatly to just below the elbows. Top two buttons open, remaining buttons closed. Natural fabric texture and realistic folds. Preserve the specified color accurately under the lighting.

TROUSERS
{trouser_color} ({trouser_hex}) tailored formal trousers in smooth lightweight wool-blend suiting fabric. Modern tailored fit. Medium-high rise. Clean flat front. Natural tailoring and realistic fabric tension.

BELT
{belt_color} ({belt_hex}) simple leather dress belt with a simple rectangular brushed-metal buckle.

SHOES
{shoe_color} ({shoe_hex}) classic {shoe_type} dress shoes in full-grain polished leather with subtle natural reflections. Realistic leather texture.

WATCH
Stainless-steel dress wristwatch with a {dial_color} ({dial_hex}) dial and a {strap_color} ({strap_hex}) leather strap.

POSE
Relaxed, confident business-fashion posture. Both hands inside the trouser pockets. One leg crossed naturally in front of the other at the ankles. Natural weight distribution.

ENVIRONMENT
Upscale modern office interior. Contemporary understated architecture. Warm beige and taupe tones. Softly blurred background with realistic depth.

LIGHTING
Soft diffused indoor lighting. Warm neutral color temperature. Natural-looking shadows. Realistic highlights on leather, metal, and fabric.

CAMERA & PHOTOGRAPHY
Premium commercial menswear photography. DSLR photographic realism. Lens approximately 85mm. Straight-on perspective, slightly above waist height. Moderate shallow depth of field. Subject sharply focused. Background softly blurred. Vertical fashion editorial composition."""

NEGATIVE_PROMPT = (
    "blazer, jacket, tie, pocket square, jeans, sneakers, shorts, "
    "oversized clothing, exaggerated muscles, distorted anatomy, extra fingers, "
    "deformed hands, floating garments, unrealistic fabric folds, "
    "artificial-looking skin, excessive HDR, extreme contrast, dramatic fashion pose, "
    "visible full face, logos, brand names, watermark, text, cartoon, anime, "
    "illustration, painting, drawing, low quality, blurry"
)


def assemble_prompt(input_json):
    return PROMPT_TEMPLATE.format(**input_json)


# Test with first combination
sample = combinations[0]['input_json']
prompt = assemble_prompt(sample)
print(f'Prompt for {sample["combo_id"]}:')
print(prompt[:300] + '...')
print(f'\nPrompt length: {len(prompt)} chars, ~{len(prompt.split())} words')

In [ ]:
# Cell 4: Load PixArt-α
import torch
from diffusers import PixArtAlphaPipeline

pipe = PixArtAlphaPipeline.from_pretrained(
    "PixArt-alpha/PixArt-XL-2-1024-MS",
    torch_dtype=torch.float16,
    use_safetensors=True,
)
pipe.to("cuda")

print('PixArt-α loaded successfully')

In [ ]:
# Cell 5: Generate a single test image (L-001)
import os

os.makedirs('/content/output', exist_ok=True)

test = combinations[0]
prompt = assemble_prompt(test['input_json'])

image = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    height=1344,
    width=768,
    num_inference_steps=30,
    guidance_scale=4.5,
).images[0]

path = f'/content/output/{test["combo_id"]}.png'
image.save(path)
print(f'Saved: {path}')

from IPython.display import display
display(image)

In [ ]:
# Cell 6: Batch generate all 154 images
import os
from tqdm.notebook import tqdm

os.makedirs('/content/output', exist_ok=True)

generated = []
skipped = []
failed = []

for combo in tqdm(combinations, desc='Generating images'):
    combo_id = combo['combo_id']
    path = f'/content/output/{combo_id}.png'

    if os.path.exists(path):
        skipped.append(combo_id)
        continue

    try:
        prompt = assemble_prompt(combo['input_json'])
        image = pipe(
            prompt=prompt,
            negative_prompt=NEGATIVE_PROMPT,
            height=1344,
            width=768,
            num_inference_steps=30,
            guidance_scale=4.5,
        ).images[0]
        image.save(path)
        generated.append(combo_id)
    except Exception as e:
        failed.append((combo_id, str(e)))
        print(f'FAILED {combo_id}: {e}')

print(f'\nDone! Generated: {len(generated)}, Skipped: {len(skipped)}, Failed: {len(failed)}')
if failed:
    print('Failed IDs:', [f[0] for f in failed])

In [ ]:
# Cell 7: Download all images as zip
import zipfile
import os
from google.colab import files

zip_path = '/content/wardrobe_images.zip'
output_dir = '/content/output'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir(output_dir)):
        if fname.endswith('.png'):
            zf.write(os.path.join(output_dir, fname), fname)

print(f'Zipped {len([f for f in os.listdir(output_dir) if f.endswith(".png")])} images')
files.download(zip_path)